# Chest X-ray Classification with PyTorch (ResNet-18)

Educational transfer-learning project adapted from a completed DataCamp project. Binary NORMAL vs PNEUMONIA classification. **Not for clinical use.**

The ZIP exported from DataCamp included X-ray images; they are intentionally not republished in this repository. Supply your own appropriately licensed copy in `data/chestxrays/{train,test}/{NORMAL,PNEUMONIA}/` before running.


## Imports and reproducibility

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report

SEED = 101010
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


## Load and preprocess data

In [ ]:
# Define the transformations to apply to the images for use with ResNet-18
transform_mean = [0.485, 0.456, 0.406]
transform_std =[0.229, 0.224, 0.225]
transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize(mean=transform_mean, std=transform_std)])

# Apply the image transforms
train_dataset = ImageFolder('data/chestxrays/train', transform=transform)
test_dataset = ImageFolder('data/chestxrays/test', transform=transform)

# Hold out test data; split the training set before training.
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


## Stratified train/validation split

In [82]:

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
import numpy as np

# Get original training labels
labels = np.array(train_dataset.targets)
indices = np.arange(len(train_dataset))

# Create a stratified 80/20 split
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

# Create dataset subsets
train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(train_dataset, val_idx)

# Create new DataLoaders
train_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False
)

# Verify class distributions
print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))

print(
    "Training class counts:",
    np.bincount(labels[train_idx])
)

print(
    "Validation class counts:",
    np.bincount(labels[val_idx])
)

print("Test images (untouched):", len(test_dataset))


Training images: 240
Validation images: 60
Training class counts: [120 120]
Validation class counts: [30 30]
Test images (untouched): 100


## Three-epoch transfer learning

In [83]:

# Load a fresh pretrained ResNet-18 model
resnet18 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Freeze all pretrained layers
for param in resnet18.parameters():
    param.requires_grad = False

# Replace the final classification layer
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 1)

# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    resnet18.fc.parameters(),
    lr=0.0003
)

# Track training and validation losses
train_losses = []
val_losses = []

# Train for 3 epochs
for epoch in range(3):

    # Keep frozen backbone in evaluation mode
    resnet18.eval()
    resnet18.fc.train()

    running_loss = 0.0

    for inputs, labels in train_loader:

        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()

        outputs = resnet18(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    avg_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(avg_train_loss)

    # Validate the model
    resnet18.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in val_loader:

            labels = labels.float().unsqueeze(1)

            outputs = resnet18(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)

    avg_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

print("Training and validation completed!")


Epoch 1/3 | Train Loss: 0.6415 | Val Loss: 0.5809
Epoch 2/3 | Train Loss: 0.5697 | Val Loss: 0.5170
Epoch 3/3 | Train Loss: 0.5107 | Val Loss: 0.4689
Training and validation completed!


## Validation evaluation

In [84]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report
)

# Evaluate on validation data only
resnet18.eval()

val_predictions = []
val_true_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = resnet18(inputs)
        predictions = (torch.sigmoid(outputs) >= 0.5).int()

        val_predictions.extend(predictions.view(-1).tolist())
        val_true_labels.extend(labels.view(-1).tolist())

# Calculate validation metrics
val_accuracy = accuracy_score(val_true_labels, val_predictions)
val_f1 = f1_score(val_true_labels, val_predictions)
val_recall = recall_score(val_true_labels, val_predictions)
val_precision = precision_score(val_true_labels, val_predictions)

print(f"Validation Accuracy: {val_accuracy:.3f}")
print(f"Validation F1-Score: {val_f1:.3f}")
print(f"Pneumonia Recall: {val_recall:.3f}")
print(f"Pneumonia Precision: {val_precision:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(val_true_labels, val_predictions))

print("\nClassification Report:")
print(
    classification_report(
        val_true_labels,
        val_predictions,
        target_names=["NORMAL", "PNEUMONIA"],
        zero_division=0
    )
)


Validation Accuracy: 0.900
Validation F1-Score: 0.893
Pneumonia Recall: 0.833
Pneumonia Precision: 0.962

Confusion Matrix:
[[29  1]
 [ 5 25]]

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.85      0.97      0.91        30
   PNEUMONIA       0.96      0.83      0.89        30

    accuracy                           0.90        60
   macro avg       0.91      0.90      0.90        60
weighted avg       0.91      0.90      0.90        60



## Save trained checkpoint (local only; ignored by Git)

In [85]:

import torch

# Save current model weights and experiment details
torch.save({
    "model_state_dict": resnet18.state_dict(),
    "validation_accuracy": val_accuracy,
    "validation_f1": val_f1,
    "validation_recall": val_recall,
    "validation_precision": val_precision,
    "epochs": 3,
    "learning_rate": 0.0003,
    "batch_size": 16,
    "train_size": 240,
    "validation_size": 60
}, "resnet18_xray_validation90.pth")

print("Model checkpoint saved successfully!")


Model checkpoint saved successfully!


## Exploratory test evaluation

**Caution:** Earlier iterations of this project used the test set for experimentation. This test result is not an untouched independent benchmark.

In [86]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report
)

# Evaluate the newly trained model
resnet18.eval()

test_predictions = []
test_true_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = resnet18(inputs)
        predictions = (torch.sigmoid(outputs) >= 0.5).int()

        test_predictions.extend(predictions.view(-1).tolist())
        test_true_labels.extend(labels.view(-1).tolist())

# Calculate test metrics
test_accuracy = accuracy_score(
    test_true_labels, test_predictions
)
test_f1_score = f1_score(
    test_true_labels, test_predictions
)

test_recall = recall_score(
    test_true_labels, test_predictions
)
test_precision = precision_score(
    test_true_labels, test_predictions
)

print(f"Test Accuracy: {test_accuracy:.3f}")
print(f"Test F1-Score: {test_f1_score:.3f}")
print(f"Pneumonia Recall: {test_recall:.3f}")
print(f"Pneumonia Precision: {test_precision:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(test_true_labels, test_predictions))

print("\nClassification Report:")
print(
    classification_report(
        test_true_labels,
        test_predictions,
        target_names=["NORMAL", "PNEUMONIA"],
        zero_division=0
    )
)


Test Accuracy: 0.800
Test F1-Score: 0.821
Pneumonia Recall: 0.920
Pneumonia Precision: 0.742

Confusion Matrix:
[[34 16]
 [ 4 46]]

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.89      0.68      0.77        50
   PNEUMONIA       0.74      0.92      0.82        50

    accuracy                           0.80       100
   macro avg       0.82      0.80      0.80       100
weighted avg       0.82      0.80      0.80       100

